# Notebook 7: Urban Planning — Low-Emission Zone Impact Analysis
## *Measure before/after air quality changes across city districts*

**Target audience:** City planners, municipalities, mobility consultants, environmental impact assessors

**Use cases:**
- Evaluate effectiveness of Low-Emission Zones (LEZ), congestion charges, cycling infrastructure
- Compare districts for school placement, park investment, or zoning decisions
- Produce data-backed evidence for funding applications and public consultations

### What we build
1. **Before/after analysis** — 5 years pre vs 5 years post a policy intervention
2. **District comparison** — heat map of pollution across a city's neighbourhoods
3. **Hotspot identification** — find areas where WHO limits are most exceeded

> We use London's Ultra Low Emission Zone (ULEZ), expanded city-wide in August 2023, as a case study.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from jiskta import JisktaClient

API_KEY = os.environ.get('JISKTA_API_KEY', 'your_api_key_here')
client = JisktaClient(API_KEY)

WHO_NO2  = 10.0   # µg/m³  WHO 2021 annual guideline
EU_NO2   = 40.0   # µg/m³  current EU AQD annual limit
EU30_NO2 = 20.0   # µg/m³  EU AQD 2024 revision (in force 2030)

print('Ready.')

## 1. Before/After: London ULEZ Expansion (2023)

London's ULEZ was extended to the full Greater London boundary on **29 August 2023**.
We compare annual NO₂ for the 3 years before (2020–2022) vs the year of expansion (2023)
across inner, middle, and outer London.

In [ ]:
# Three zones: inner, middle, outer London
london_zones = {
    'Inner London\n(Zone 1–2)':   (51.45, 51.55, -0.15, 0.00),
    'Middle London\n(Zone 3–4)':  (51.43, 51.55, -0.30, -0.15),
    'Outer London\n(Zone 5–6)':   (51.35, 51.45, -0.40, -0.25),
}

# Query annual means for 2019–2023
years = [2019, 2020, 2021, 2022, 2023]
results = {zone: [] for zone in london_zones}

for year in years:
    for zone, (lat_min, lat_max, lon_min, lon_max) in london_zones.items():
        df = client.query(
            lat=(lat_min, lat_max), lon=(lon_min, lon_max),
            start=f'{year}-01', end=f'{year}-12',
            variables=['no2'],
            aggregate='monthly',
        )
        annual = df['no2_mean'].mean()
        results[zone].append(annual)
    print(f'{year}: {" | ".join(f"{z.split(chr(10))[0]}: {results[z][-1]:.2f}" for z in london_zones)}')

print(f'\nTotal cost: {len(years) * len(london_zones)} credits')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
zone_colors = ['#1D4ED8', '#7C3AED', '#BE185D']

for (zone, vals), color in zip(results.items(), zone_colors):
    label = zone.replace('\n', ' ')
    ax.plot(years, vals, 'o-', label=label, color=color,
            linewidth=2.5, markersize=8)

# ULEZ expansion marker
ax.axvline(2023, color='#DC2626', linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(2023.05, ax.get_ylim()[1] * 0.98, 'ULEZ\nexpansion\nAug 2023',
        fontsize=8, color='#DC2626', va='top')

# Guidelines
ax.axhline(WHO_NO2, color='#16A34A', linestyle=':', linewidth=1, alpha=0.7)
ax.text(years[0] - 0.05, WHO_NO2 + 0.3, f'WHO 2021 ({WHO_NO2} µg/m³)',
        fontsize=7.5, color='#16A34A', ha='left')
ax.axhline(EU30_NO2, color='#CA8A04', linestyle=':', linewidth=1, alpha=0.7)
ax.text(years[0] - 0.05, EU30_NO2 + 0.3, f'EU AQD 2030 ({EU30_NO2} µg/m³)',
        fontsize=7.5, color='#CA8A04', ha='left')

ax.set_xlabel('Year')
ax.set_ylabel('Annual mean NO₂ (µg/m³)')
ax.set_title('London NO₂ Trend 2019–2023 by Zone\n'
             'ULEZ expanded to full Greater London boundary in August 2023', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.2)
ax.set_xticks(years)
plt.tight_layout()
plt.savefig('../exports/07_london_ulez_trend.png', dpi=150, bbox_inches='tight')
plt.show()

# Year-on-year change for 2022→2023
print('\nYear-on-year change 2022→2023:')
for zone, vals in results.items():
    delta = vals[-1] - vals[-2]
    pct = delta / vals[-2] * 100
    print(f'  {zone.split(chr(10))[0]:<25} {delta:+.2f} µg/m³  ({pct:+.1f}%)')

## 2. District Comparison Heatmap

Compare pollution levels across a grid of city districts — useful for school placement,
park investment prioritisation, or resident complaints responses.

In [ ]:
# Query a 0.5°×0.5° grid around Paris at 0.1° resolution
# using the stats format — one API call returns all grid points
df_grid = client.query(
    lat=(48.75, 49.05),
    lon=(2.10, 2.60),
    start='2023-01',
    end='2023-12',
    variables=['no2'],
    aggregate='monthly',
)

# Annual mean per grid point
annual = df_grid.groupby(['lat', 'lon'])['no2_mean'].mean().reset_index()
annual.columns = ['lat', 'lon', 'no2_annual']

# Pivot to 2D grid
heatmap = annual.pivot(index='lat', columns='lon', values='no2_annual')
print(f'Grid: {heatmap.shape[0]} lat × {heatmap.shape[1]} lon points')
print(f'Range: {annual["no2_annual"].min():.2f} – {annual["no2_annual"].max():.2f} µg/m³')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Left: absolute NO2 heatmap ----
ax = axes[0]
im = ax.imshow(heatmap.values, origin='lower', aspect='auto',
               cmap='RdYlGn_r', vmin=5, vmax=30,
               extent=[heatmap.columns.min(), heatmap.columns.max(),
                       heatmap.index.min(),  heatmap.index.max()])
plt.colorbar(im, ax=ax, label='Annual mean NO₂ (µg/m³)', shrink=0.8)
ax.set_title('Paris NO₂ Grid — 2023\n(0.1° resolution, CAMS reanalysis)', fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

# ---- Right: WHO exceedance factor (multiples of guideline) ----
ax2 = axes[1]
exceedance = heatmap / WHO_NO2
im2 = ax2.imshow(exceedance.values, origin='lower', aspect='auto',
                 cmap='OrRd', vmin=0.5, vmax=4,
                 extent=[heatmap.columns.min(), heatmap.columns.max(),
                         heatmap.index.min(),  heatmap.index.max()])
plt.colorbar(im2, ax=ax2, label='× WHO 2021 guideline (10 µg/m³)', shrink=0.8)
ax2.set_title('WHO Exceedance Factor — 2023\n1× = at guideline, 2× = double the limit', fontsize=11)
ax2.set_xlabel('Longitude'); ax2.set_ylabel('Latitude')

plt.suptitle('Greater Paris Air Quality Map — 2023 Annual NO₂', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../exports/07_paris_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Hotspot Identification — Where Do WHO Limits Get Exceeded?

Find the worst grid points and flag them for intervention priority.

In [ ]:
# Rank all grid points
ranked = annual.sort_values('no2_annual', ascending=False).copy()
ranked['who_multiple'] = (ranked['no2_annual'] / WHO_NO2).round(2)
ranked['eu_2030_status'] = ranked['no2_annual'].apply(
    lambda x: '✅ Compliant' if x <= EU30_NO2 else '⚠️ Non-compliant'
)
ranked['priority'] = pd.cut(
    ranked['no2_annual'],
    bins=[0, WHO_NO2, EU30_NO2, EU_NO2, 999],
    labels=['Low', 'Medium', 'High', 'Critical']
)

print('Top 10 highest-NO₂ grid points:')
display(ranked.head(10).reset_index(drop=True))

print(f'\nPriority breakdown:')
print(ranked['priority'].value_counts().sort_index())

## 4. Multi-year improvement trend — city-wide

Track the annual mean for the full city to report on long-term air quality improvement.
Each year = 1 API call (all grid points in one request).

In [ ]:
city_trend = {}
for year in range(2015, 2024):
    df_y = client.query(
        lat=(48.75, 49.05), lon=(2.10, 2.60),
        start=f'{year}-01', end=f'{year}-12',
        variables=['no2'], aggregate='monthly',
    )
    city_trend[year] = df_y['no2_mean'].mean()
    print(f'{year}: {city_trend[year]:.2f} µg/m³')

print(f'\nTotal improvement {2015}→{2023}: '
      f'{city_trend[2023] - city_trend[2015]:+.2f} µg/m³ '
      f'({(city_trend[2023]/city_trend[2015]-1)*100:+.1f}%)')
print(f'Total cost: {9} credits (1 per year)')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
xs = list(city_trend.keys())
ys = list(city_trend.values())

ax.fill_between(xs, ys, alpha=0.15, color='#2563EB')
ax.plot(xs, ys, 'o-', color='#2563EB', linewidth=2.5, markersize=8, label='City mean NO₂')

# Trendline
z = np.polyfit(xs, ys, 1)
ax.plot(xs, np.poly1d(z)(xs), '--', color='#2563EB', alpha=0.5, linewidth=1.5,
        label=f'Trend ({z[0]:+.2f} µg/m³/yr)')

for limit, label, color in [
    (WHO_NO2,  f'WHO 2021 ({WHO_NO2} µg/m³)',  '#16A34A'),
    (EU30_NO2, f'EU AQD 2030 ({EU30_NO2} µg/m³)', '#CA8A04'),
]:
    ax.axhline(limit, linestyle=':', color=color, linewidth=1.5, label=label)

ax.set_xlabel('Year'); ax.set_ylabel('Annual mean NO₂ (µg/m³)')
ax.set_title('Paris City-Wide NO₂ Trend 2015–2023\n'
             'Each data point = mean of all grid cells in bbox (1 API call/year)', fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.2); ax.set_xticks(xs)
plt.tight_layout()
plt.savefig('../exports/07_citywide_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Diurnal Pattern — When are peaks highest?

Understanding the daily cycle helps planners design time-based interventions
(e.g., school drop-off bans, loading bay time restrictions).

In [ ]:
# Diurnal aggregate: returns 24 hourly means over the full query period
df_diurnal = client.query(
    lat=(48.75, 49.05), lon=(2.10, 2.60),
    start='2023-01', end='2023-12',
    variables=['no2'],
    aggregate='diurnal',
)

# Average across grid points
diurnal_mean = df_diurnal.groupby('hour')['no2_mean'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(diurnal_mean.index, diurnal_mean.values, alpha=0.2, color='#2563EB')
ax.plot(diurnal_mean.index, diurnal_mean.values, 'o-', color='#2563EB',
        linewidth=2.5, markersize=5)

# Highlight rush hours
for start, end, label in [(7, 10, 'Morning rush'), (17, 20, 'Evening rush')]:
    ax.axvspan(start, end, alpha=0.08, color='#DC2626')
    ax.text((start + end)/2, diurnal_mean.max() * 0.98, label,
            ha='center', fontsize=8, color='#DC2626')

ax.set_xlabel('Hour of day (UTC)')
ax.set_ylabel('Mean NO₂ (µg/m³)')
ax.set_title('Paris Diurnal NO₂ Pattern — 2023 Annual Average\n'
             'Averaged over full city bbox · Shaded = typical rush hours', fontsize=12)
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('../exports/07_diurnal.png', dpi=150, bbox_inches='tight')
plt.show()

peak_hour = diurnal_mean.idxmax()
trough_hour = diurnal_mean.idxmin()
print(f'Peak hour: {peak_hour:02d}:00 UTC ({diurnal_mean[peak_hour]:.2f} µg/m³)')
print(f'Trough hour: {trough_hour:02d}:00 UTC ({diurnal_mean[trough_hour]:.2f} µg/m³)')
print(f'Peak/trough ratio: {diurnal_mean[peak_hour]/diurnal_mean[trough_hour]:.2f}×')